# 🌡️ Prevendo Temperatura com Redes Neurais (Regressão)

Este notebook segue o mesmo espírito **passo a passo** do notebook de classificação (flores do dataset Iris), mas agora resolvendo um problema diferente: **regressão**.

Em vez de escolher uma categoria (espécie de flor), a rede neural vai prever um **número contínuo**: a temperatura média de um mês, a partir do histórico de meses anteriores. Esse é exatamente o tipo de problema trabalhado na **2ª Lista de Exercícios** (previsão de séries temporais com MLP), então vamos reaproveitar as mesmas ideias — janela de dados (*lag*), variável exógena (mês) e previsão *one-step* — só que explicadas de forma bem gradual.

## 0. Preparando o ambiente

Vamos usar as mesmas bibliotecas do notebook de classificação:

- **pandas**: para carregar e explorar os dados em formato de tabela;
- **matplotlib**: para visualizar a série temporal e os resultados;
- **scikit-learn (sklearn)**: para normalizar os dados e calcular métricas de erro;
- **torch (PyTorch)**: para construir e treinar a rede neural.

Se alguma dessas bibliotecas não estiver instalada, rode no terminal: `pip install torch pandas scikit-learn matplotlib`.

In [ ]:
import torch
import torch.nn as nn                              # módulo para definir a rede neural
from torch.utils.data import Dataset, DataLoader   # utilidades para organizar os dados

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Fixamos uma "semente" aleatória para que os resultados sejam sempre os mesmos quando você rodar de novo
torch.manual_seed(42)

## 1. Conhecendo o dataset

Vamos usar a série `microclima1`, a mesma da 2ª Lista de Exercícios. Ela traz a temperatura média mensal de uma estação, registrada ao longo de vários anos. O arquivo tem duas colunas:

- `value`: a temperatura média do mês (o que queremos prever);
- `time`: o mês do ano, de `1` (janeiro) a `12` (dezembro) — essa é a nossa **variável exógena**, isto é, uma informação extra (além do histórico da própria série) que ajuda a rede a prever melhor, já que sabemos que a temperatura tem um comportamento sazonal (repete-se a cada 12 meses).

**Nosso objetivo**: dado o histórico dos últimos meses (e o mês que queremos prever), a rede neural deve estimar a temperatura média daquele mês. Isso é um problema de **regressão** (prever um número contínuo), diferente de **classificação** (escolher entre categorias).

In [ ]:
dataset_name = 'microclima1'
path = f'https://raw.githubusercontent.com/mdrs-thiago/PUC_Redes_Neurais/main/datasets/s_{dataset_name}.csv'
df = pd.read_csv(path)

df.head()

### Dando uma olhada geral nos dados

Antes de sair treinando redes neurais, é sempre bom **visualizar a série**. Perguntas úteis:

- A série tem um padrão sazonal (que se repete todo ano)?
- Existe alguma tendência de crescimento ou queda ao longo do tempo?
- Qual a escala de valores da série?

In [ ]:
df.describe()

In [ ]:
# Visualizando a dinâmica temporal da série completa
plt.figure(figsize=(10, 4))
plt.plot(df['value'].values)
plt.title('Temperatura média mensal — microclima1')
plt.xlabel('Mês (índice sequencial)')
plt.ylabel('Temperatura')
plt.grid(True)
plt.show()

## 2. Preparando os dados para a rede neural

Diferente do dataset Iris (em que cada linha já era um exemplo pronto), uma série temporal **não vem em formato tabular**. Precisamos transformá-la antes de treinar a rede. Vamos seguir a mesma lógica da 2ª Lista de Exercícios:

1. **Criar uma janela deslizante (*lag*)**: para prever o valor do mês atual `y(t)`, usamos os `lag` valores anteriores da série como entrada: `y(t-lag), ..., y(t-1)`;
2. **Adicionar a variável exógena**: incluímos o mês `t` que queremos prever (`1` a `12`), pois ele carrega a informação sazonal;
3. **Normalizar** tudo para a rede aprender melhor, exatamente como fizemos com as medidas das flores.

### 2.1 Criando a janela deslizante (*lag*)

A função abaixo desliza uma janela de tamanho `lag` sobre a série: cada linha do novo dataframe contém `lag` valores passados, o mês a ser previsto, e o valor real daquele mês (o alvo `y(t)`).

In [ ]:
def create_windowed_dataset(data, lag=12):
    '''
    Transforma a série temporal (formato de coluna única) em uma tabela, onde cada linha
    contém os `lag` valores anteriores da série (colunas y(t-lag) ... y(t-1)), o mês a ser
    previsto (coluna 'mes') e o valor real daquele mês (coluna y(t)).
    '''
    data = data.copy()

    for i in range(lag, 0, -1):
        data[f'y(t-{i})'] = data['value'].shift(i)

    data['mes'] = data['time']
    data['y(t)'] = data['value']

    # As primeiras `lag` linhas não têm histórico suficiente (shift gera valores NaN) e são descartadas
    data = data.dropna().reset_index(drop=True)

    lag_columns = [f'y(t-{i})' for i in range(lag, 0, -1)]
    return data[lag_columns + ['mes', 'y(t)']]


lag = 12
windowed_df = create_windowed_dataset(df, lag=lag)
windowed_df.head()

### 2.2 Dividindo em treino, validação e teste

⚠️ **Atenção, diferença importante em relação à classificação!** No dataset Iris, embaralhamos os dados livremente antes de dividir, pois cada flor era um exemplo independente. Aqui **não podemos embaralhar**: os exemplos têm uma ordem temporal, e usar meses "do futuro" para prever meses "do passado" (ou vice-versa) vazaria informação e daria uma falsa sensação de que o modelo está indo bem.

Por isso, para séries temporais, separamos os conjuntos **respeitando a ordem cronológica**: os primeiros meses ficam para treino, os seguintes para validação, e os últimos (mais recentes) para teste — simulando a situação real de prever o futuro.

In [ ]:
n = len(windowed_df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = windowed_df.iloc[:train_end]
val_df = windowed_df.iloc[train_end:val_end]
test_df = windowed_df.iloc[val_end:]

print('Exemplos de treino    :', len(train_df))
print('Exemplos de validação :', len(val_df))
print('Exemplos de teste     :', len(test_df))

### 2.3 Separando entradas (X) e saída (y), e normalizando

Assim como fizemos com as flores, separamos as colunas de entrada (`X`, os valores passados + o mês) da coluna de saída (`y`, o valor a ser previsto), e normalizamos tudo para o intervalo `[0, 1]` com o `MinMaxScaler`.

⚠️ Aqui a regra continua a mesma de antes: aprendemos (`fit`) o mínimo e o máximo **apenas com os dados de treino**, e aplicamos (`transform`) essa mesma regra na validação e no teste, para evitar *data leakage*.

Uma diferença importante em relação à classificação: como o alvo `y` agora é um **número contínuo** (e não uma categoria), também precisamos normalizá-lo — por isso usamos **dois** `MinMaxScaler`: um para as entradas (`X`) e outro para a saída (`y`). Guardamos o `scaler_y` porque, no final, vamos precisar **desnormalizar** as previsões para interpretá-las na escala real de temperatura.

In [ ]:
feature_columns = [c for c in windowed_df.columns if c != 'y(t)']

X_train, y_train = train_df[feature_columns].values, train_df[['y(t)']].values
X_val, y_val = val_df[feature_columns].values, val_df[['y(t)']].values
X_test, y_test = test_df[feature_columns].values, test_df[['y(t)']].values

scaler_X = MinMaxScaler()
X_train_std = scaler_X.fit_transform(X_train)
X_val_std = scaler_X.transform(X_val)
X_test_std = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_std = scaler_y.fit_transform(y_train)
y_val_std = scaler_y.transform(y_val)
y_test_std = scaler_y.transform(y_test)

## 3. Organizando os dados: `Dataset` e `DataLoader`

Assim como na classificação, criamos um `Dataset` customizado para entregar `X` e `y` já convertidos em tensores, e um `DataLoader` para servir os dados em lotes durante o treino.

A única diferença é o tipo do alvo `y`: na classificação, `y` era um índice inteiro (`long`) representando a classe. Aqui, `y` é um **valor contínuo** (`float`), então convertemos ambos, `X` e `y`, para tensores de ponto flutuante.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = X_data
        self.y_data = y_data

    def __getitem__(self, index):
        X_batch = torch.from_numpy(self.X_data[index]).float()
        y_batch = torch.from_numpy(self.y_data[index]).float()
        return X_batch, y_batch

    def __len__(self):
        return len(self.X_data)


train_dataset = TimeSeriesDataset(X_train_std, y_train_std)
val_dataset = TimeSeriesDataset(X_val_std, y_val_std)
test_dataset = TimeSeriesDataset(X_test_std, y_test_std)

# Vamos espiar um único exemplo para ver o formato
X_exemplo, y_exemplo = train_dataset[0]
print('Entrada (histórico + mês, normalizados):', X_exemplo)
print('Saída (temperatura normalizada)        :', y_exemplo)

In [ ]:
# Para séries temporais, mantemos shuffle=False: a ordem dos exemplos dentro de cada conjunto já não
# importa tanto para o treino em si (cada linha já resume uma janela do passado), mas evitamos qualquer
# mistura entre os conjuntos de treino, validação e teste.
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

## 4. Montando a rede neural

A arquitetura é muito parecida com a do classificador:

```
entrada (lag valores passados + mês) → camada oculta (ativação ReLU) → saída (1 valor previsto)
```

A diferença principal está na **camada de saída**: no classificador, tínhamos 3 neurônios de saída (um "placar" por espécie) que depois interpretávamos com `softmax`. Na regressão, temos **apenas 1 neurônio de saída**, e ele **não passa por nenhuma função de ativação** — o valor bruto (linear) já é a própria previsão de temperatura (normalizada).

In [ ]:
class TemperatureRegressor(nn.Module):
    def __init__(self, n_in, hidden_size=32):
        super().__init__()

        self.fc1 = nn.Linear(in_features=n_in, out_features=hidden_size)   # camada de entrada -> oculta
        self.activation = nn.ReLU()                                        # não-linearidade
        self.fc2 = nn.Linear(in_features=hidden_size, out_features=1)      # camada oculta -> saída (1 valor)

    def forward(self, x):
        h1 = self.fc1(x)          # combinação linear das entradas
        a1 = self.activation(h1)  # aplica a não-linearidade
        out = self.fc2(a1)        # gera 1 número: a previsão de temperatura (normalizada)
        return out


n_in = X_train_std.shape[1]
model = TemperatureRegressor(n_in=n_in, hidden_size=32)
model

## 5. Função de perda e otimizador

Para classificação usamos `CrossEntropyLoss`, que compara probabilidades entre categorias. Para regressão, o mais comum é medir o quão distante a previsão ficou do valor real usando o **erro quadrático médio**:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

No PyTorch, isso é a função `nn.MSELoss()`. O otimizador continua sendo o `Adam`.

In [ ]:
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 6. Treinando a rede

O laço de treino é idêntico, em estrutura, ao do classificador: para cada lote de dados, zeramos os gradientes, calculamos a previsão, medimos o erro (agora com `MSELoss`), retropropagamos e damos um passo do otimizador. Ao final de cada época, também acompanhamos a perda no conjunto de validação.

In [ ]:
epochs = 200
historico_perda_treino = []
historico_perda_val = []

for epoch in range(epochs):
    # --- Treino ---
    model.train()
    perda_epoca = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        y_hat = model(X_batch)
        loss = loss_function(y_hat, y_batch)

        loss.backward()
        optimizer.step()

        perda_epoca += loss.item()

    historico_perda_treino.append(perda_epoca / len(train_loader))

    # --- Validação ---
    model.eval()
    perda_val_epoca = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            y_hat = model(X_batch)
            loss = loss_function(y_hat, y_batch)
            perda_val_epoca += loss.item()

    historico_perda_val.append(perda_val_epoca / len(val_loader))

    if (epoch + 1) % 20 == 0:
        print(f'Época [{epoch+1}/{epochs}] - Loss treino: {historico_perda_treino[-1]:.4f} '
              f'- Loss validação: {historico_perda_val[-1]:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historico_perda_treino, label='Treino')
plt.plot(historico_perda_val, label='Validação')
plt.title('Loss ao longo do treinamento')
plt.xlabel('Época')
plt.ylabel('Loss (MSELoss)')
plt.legend()
plt.grid(True)
plt.show()

## 7. Avaliando o modelo

Assim como no classificador, avaliamos o modelo no conjunto de **teste** (meses que ele nunca viu). Mas as métricas mudam: em vez de precisão/recall (que comparam categorias), usamos métricas de erro para valores contínuos:

- **MAE (Erro Absoluto Médio)**: em média, o quanto a previsão erra, na mesma unidade da temperatura;
- **RMSE (Raiz do Erro Quadrático Médio)**: parecido com o MAE, mas penaliza mais os erros grandes.

⚠️ Antes de calcular as métricas, precisamos **desnormalizar** as previsões (`scaler_y.inverse_transform`) para que o erro seja interpretável na escala real de temperatura, e não no intervalo `[0, 1]`.

In [ ]:
model.eval()

with torch.no_grad():
    X_test_tensor = torch.from_numpy(X_test_std).float()
    y_hat_std = model(X_test_tensor).numpy()

# Desnormalizando para a escala real de temperatura
y_pred = scaler_y.inverse_transform(y_hat_std)
y_real = scaler_y.inverse_transform(y_test_std)

mae_error = mean_absolute_error(y_real, y_pred)
mse_error = mean_squared_error(y_real, y_pred)
rmse_error = np.sqrt(mse_error)

print(f'Erro MAE  = {mae_error:.3f}')
print(f'Erro MSE  = {mse_error:.3f}')
print(f'Erro RMSE = {rmse_error:.3f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_real, label='Real')
plt.plot(y_pred, label='Previsto')
plt.title('Temperatura real vs. prevista — conjunto de teste')
plt.xlabel('Mês (conjunto de teste)')
plt.ylabel('Temperatura')
plt.legend()
plt.grid(True)
plt.show()

## 8. Previsão *one-step*: estimando o próximo mês

Por fim, vamos simular o uso real do modelo: temos o histórico dos últimos `lag` meses e queremos prever a temperatura do próximo mês. Isso é chamado de previsão ***one-step***, pois prevemos apenas **um passo à frente** (diferente da previsão *multi-step*, trabalhada na 2ª Lista de Exercícios, em que a própria previsão é reaproveitada como entrada para prever vários meses seguidos).

Repare que aplicamos **os mesmos `scaler_X` e `scaler_y`** usados no treino, para manter a mesma escala de valores, e desnormalizamos a saída no final para interpretar o resultado.

In [ ]:
# Usamos a última janela disponível no conjunto de teste como exemplo
ultima_janela = X_test[-1:]
mes_previsto_real = y_test[-1, 0]

ultima_janela_std = scaler_X.transform(ultima_janela)

model.eval()
with torch.no_grad():
    entrada = torch.from_numpy(ultima_janela_std).float()
    previsao_std = model(entrada).numpy()

previsao = scaler_y.inverse_transform(previsao_std)

print('Temperatura prevista para o próximo mês:', round(previsao[0, 0], 2))
print('Temperatura real (para comparação)     :', round(mes_previsto_real, 2))